# Compartment - mmBERT pipeline (Kaggle)

Chạy toàn bộ M3 chain trong một session: **probe → warmup → train5 → predict**.

### Lấy repo (2 cách, ưu tiên mount trước)
1. **Kaggle dataset mount**: repo nằm dưới `/kaggle/input/**/Compartment` (đoạn auto-detect). **Lưu ý**: dataset đang mount là code cũ — phải re-upload bản mới nhất (M3) mới có `mm/`.
2. **Git clone từ GitHub**: nếu không thấy path mount được, notebook clone `REPO_URL` (mặc định `AmnO-O/MoTune`, nhánh `main`) vào `/kaggle/working/Compartment`. **Bắt buộc push trước**: GitHub hiện vẫn ở commit `9e272b0` (trước M1, không có `mm/`).
- Repo **private**: đặt `GITHUB_TOKEN` (PAT) dưới dạng `os.environ['GITHUB_TOKEN']` trước khi chạy cell setup.

### Knobs (sửa dòng `MODE` bên dưới)
- `MODE = 'all'` chạy cả chain; đặt `probe | warmup | train5 | predict` để chạy riêng.
- `WARMUP_EPOCHS`, `WARMUP_BATCH` (hạ xuống 16 nếu OOM ở phase MLM).
- `MLM_DATA`: các file context dùng cho MLM warmup.
- `TRAIN5_SET`: thêm `--set` cho train5 (cách nhau bằng dấu phẩy).

Outputs ở `/kaggle/working`: `models/`, `submission/`, `metrics.json`, `history.json`, `oof_predictions.npz`, `trial_metrics.json`.
- `RESUME=false` (default): full chain (probe + warmup + train80 + predict).
- `RESUME=warmup`: tải `warmup_merged.pt` (backbone-only) từ HF rồi **bỏ warmup, train80 + predict tiếp** — warmup chỉ train backbone LM, head scorer luôn init mới nên không phụ thuộc config head.
- `RESUME=predict`: tải mọi checkpoint từ HF, chạy thẳng predict (bỏ mọi training).
Nhớ giữ OVERRIDES knobs khớp lúc train (vd `use_proto_cos`, `num_bins`, `head_mode`, `lora_from_layer`).

In [ ]:
import glob, os, subprocess, sys
from pathlib import Path

def _importable(name: str) -> bool:
    try:
        __import__(name)
        return True
    except Exception:
        return False

# ---- repo source -----------------------------------------------------------
REPO_URL = os.environ.get('REPO_URL', 'https://github.com/AmnO-O/MoTune.git')
REPO_BRANCH = os.environ.get('REPO_BRANCH', 'main')
GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN', '')    # set for private repos

# ---- always run code from a fresh GitHub clone ----------------------------
# The Kaggle-mount snapshot is frozen when the dataset is uploaded, so running
# code from the mount silently executes stale buggy versions. Data/trial are
# still read from the mount: mm.utils._find_data_dir resolves the mount dataset/
# dir via /kaggle/input/datasets/ieltsmater/compartment/Compartment, and
# _resolve_trial_path looks next to it (.../Compartment/trial).
dest = Path('/kaggle/working/Compartment')
url = REPO_URL
if GITHUB_TOKEN:
    url = url.replace('https://', f'https://{GITHUB_TOKEN}@')
dest.parent.mkdir(parents=True, exist_ok=True)
if (dest / 'run.py').is_file():
    print('refreshing existing clone at', dest)
    subprocess.run(['git', '-C', str(dest), 'fetch', '--depth', '1', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(dest), 'reset', '--hard', f'origin/{REPO_BRANCH}'], check=True)
else:
    print('cloning', REPO_URL)
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, url, str(dest)], check=True)
REPO = dest
os.chdir(REPO)
print('repo:', REPO)
print('head:', subprocess.run(['git', '-C', REPO, 'log', '--oneline', '-1'],
                              capture_output=True, text=True).stdout.strip())

# ---- knobs ----------------------------------------------------------------
MODE = os.environ.get('MODE', 'all')            # all | probe | warmup | train80 | predict
WARMUP_EPOCHS = int(os.environ.get('WARMUP_EPOCHS', '4'))
WARMUP_BATCH = os.environ.get('WARMUP_BATCH', '64')   # e.g. 16 if OOM in MLM phase
MLM_DATA = 'en-nn-train.tsv,de-nn-train.tsv,en-pv-train.tsv,de-pv-train.tsv,nctti_en.tsv'
TRAIN5_SET = os.environ.get('TRAIN5_SET', '')       # extra --set flags, comma-separated

# ---- tunable config overrides ('' = keep run.py default) -----------------
# Edit values here to tune a run without touching --set flags below.
OVERRIDES = {
    'warmup_batch_size': WARMUP_BATCH,      # e.g. 16 if OOM in MLM warmup
    'warmup_lr': '',
    'mlm_mask_span': '',                    # 'one' (mod OR head) | 'both' (whole compound)
    'mlm_mask_prob': '',
    'mlm_random_prob': '',
    'freeze_epochs': '',
    'lora_epochs': '',
    'lora_rank': '',
    'lora_alpha': '',
    'lora_from_layer': '16',                # LoRA window: top layers (0 = all)
    'batch_size': '',
    'head_lr': '',
    'encoder_lr': '',
    'ccc_weight': '',
    'lambda_rank': '',
    'lambda_compound': '',
    'num_workers': '',
    # --- Softmax Head + KL Divergence Loss ---
    'head_mode': 'gauss',                 # 'softmax' (ordinal classification) or 'reg' (regression)
    'ce_weight': '1.0',                     # Trọng số KL divergence loss (1.0 khi dùng softmax)
    'num_bins': '6',                        # Số lượng ordinal bins (thang điểm 1.0 -> 5.0)
    'bin_sigma': '0.5',                     # Độ lệch chuẩn Gaussian tạo phân phối nhãn mềm
    'predict_mode': 'single',                 # train80 -> best.pt; '5fold' needs fold0..4
    'use_proto_cos': '',                     # literality feature; must be SAME on train & predict
    'mlm_ctx_mask_ratio': '',                # 0..1 fraction of context tokens masked in MLM warmup
}

def cfg_sets(*extra):
    out = []
    for k, v in OVERRIDES.items():
        if v not in (None, ''):
            out += ['--set', f'{k}={v}']
    for e in extra:
        out += ['--set', e]
    return out


# ---- memory ------------------------------------------------------------------
# expandable segments reduce T4 fragmentation; inherited by the run.py subprocess
os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')

# ---- HuggingFace push (optional) ----------------------------------------
# 1) set Kaggle Secret 'HF_TOKEN' (your HuggingFace write token), OR
# 2) set env var HF_TOKEN before running the notebook.
# Empty HF_REPO_ID = skip push; set 'user/repo-name' to enable.
HF_REPO_ID = os.environ.get('HF_REPO_ID', 'AmnO-O/compartment-weights')
HF_PRIVATE = os.environ.get('HF_PRIVATE', 'true').lower() not in ('0','false','no')
RESUME = os.environ.get('RESUME', 'false').strip().lower()                # false | warmup | predict
                                                                           # warmup = skip MLM phase,
                                                                           # continue train80+predict
                                                                           # (warmup trains backbone only, heads are fresh)
                                                                           # predict = skip all training

def _hf_token() -> str:
    tok = os.environ.get('HF_TOKEN', '')
    if tok:
        return tok
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        return ''

# ---- ensure imports exist (Kaggle already ships them) ---------------------
for m in ('torch', 'transformers', 'pandas', 'numpy', 'sklearn', 'scipy'):
    if not _importable(m):
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', m], check=False)

def run(args):
    print('\n>>>', ' '.join(args))
    subprocess.run(args, cwd=REPO, check=True)


In [ ]:
# --- Resume from a checkpoint already pushed to HF (RESUME=true) ------
# Downloads models/*.pt + config.json from HF_REPO_ID into /kaggle/working,
# then forces predict-only (skips warmup/train). The OVERRIDES knobs below
# must match the run that produced the checkpoint (e.g. use_proto_cos).
work = Path('/kaggle/working')
RESUME = ''
if RESUME == 'true':
    RESUME = 'predict'   # back-compat: RESUME=true == predict-only
RESUMING = RESUME in ('warmup', 'predict')
if RESUMING:
    tok = _hf_token()
    if not (HF_REPO_ID and tok):
        print('resume: SKIP - need HF_REPO_ID + HF_TOKEN, will train from scratch')
        RESUME = 'false'; RESUMING = False
    else:
        try:
            from huggingface_hub import snapshot_download
            snapshot_download(repo_id=HF_REPO_ID, token=tok, local_dir=str(work))
            print('resume: downloaded HF repo', HF_REPO_ID, '->', work)
            if RESUME == 'predict':
                MODE = 'predict'      # skip warmup + train80
            else:
                MODE = MODE if MODE in ('all', 'train80') else 'all'   # warmup resume: train on
            print(f'resume: OK -> mode={RESUME} (MODE={MODE})')
        except Exception as ex:
            print(f'resume: FAILED ({ex}); will train from scratch')
            RESUME = 'false'; RESUMING = False
else:
    print(f'RESUME={RESUME} -> full chain (probe + warmup + train80 + predict)')


In [ ]:
# --- M2 gate: real-tokenizer alignment + MLM masking sanity ---------------
if MODE in ('all', 'probe') and not RESUMING:
    run([sys.executable, 'run.py', 'probe'] + cfg_sets())


In [ ]:
# --- Phase 0: compound-aware MLM warmup + LoRA merge ----------------------
# Writes /kaggle/working/models/warmup_merged.pt (used as base by train5).
if MODE in ('all', 'warmup') and not RESUMING:
    args = [sys.executable, 'run.py', 'warmup'] + cfg_sets(
        f'warmup_mlm_epochs={WARMUP_EPOCHS}',
        f'mlm_data_paths={MLM_DATA}')
    run(args)


In [ ]:
import os

# Set môi trường trực tiếp trong Python để tránh deadlock GPU và DataLoader
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# --- Phase 1: 80/20 compound split hoặc 5-fold CV -------------------------
# head_mode / ce_weight / lora_from_layer / num_bins / bin_sigma /
# predict_mode / warmup_batch_size come from OVERRIDES above (single source
# of truth, shared by warmup + train80 + predict), so only the flags NOT in
# OVERRIDES are hardcoded here. Set values via OVERRIDES or TRAIN5_SET.
if MODE in ('all', 'train80'):
    args = [
        sys.executable, 'run.py', 'train80', 
        '--set', 'num_workers=0',            # Chống deadlock Kaggle
        '--set', 'batch_size=64',            # Batch size 64
        '--set', 'freeze_epochs=3',         # Số epoch Phase 1
        '--set', 'head_lr=2e-4',             # Head Learning Rate
    ] + cfg_sets()

    if TRAIN5_SET:
        args += [x.strip() for x in TRAIN5_SET.split(',') if x.strip()]

    run(args)


In [ ]:
# --- Predict trial (5-fold ensemble) + submission -------------------------
if MODE in ('all', 'predict'):
    run([sys.executable, 'run.py', 'predict'] + cfg_sets())


In [ ]:
# --- Push trained weights to HuggingFace --------------------------------
# Needs only HF_TOKEN (Kaggle Secret or env var) + HF_REPO_ID set above.
# Skipped automatically when HF_REPO_ID is empty.
if HF_REPO_ID and MODE in ('all', 'predict'):
    if not _hf_token():
        print('SKIP push: no HF_TOKEN found (set Kaggle Secret named HF_TOKEN).')
    else:
        from huggingface_hub import HfApi
        api = HfApi(token=_hf_token())
        api.create_repo(repo_id=HF_REPO_ID, private=HF_PRIVATE, exist_ok=True, repo_type='model')
        work = Path('/kaggle/working')
        # upload all model checkpoints
        models_dir = work / 'models'
        if models_dir.is_dir():
            api.upload_folder(
                folder_path=str(models_dir),
                repo_id=HF_REPO_ID,
                path_in_repo='models',
                allow_patterns='*.pt'
            )
            pt_files = list(models_dir.glob('*.pt'))
            print(f'pushed {len(pt_files)} model file(s)')
        # upload key artifacts
        for name in ('config.json', 'metrics.json', 'trial_metrics.json',
                     'history.json', 'oof_predictions.npz'):
            src = work / name
            if src.is_file():
                api.upload_file(path_or_fileobj=str(src), path_in_repo=name, repo_id=HF_REPO_ID)
                print(f'pushed {name}')
        print(f'HF repo: https://huggingface.co/{HF_REPO_ID}')
else:
    print('push skipped (HF_REPO_ID empty or MODE=%s)' % MODE)


In [ ]:
import json
import pandas as pd

work = Path('/kaggle/working')
for name in ('metrics.json', 'trial_metrics.json'):
    p = work / name
    if p.is_file():
        print(f'--- {name} ---')
        print(json.dumps(json.loads(p.read_text(encoding='utf-8')), indent=2))

sub = work / 'submission' / 'en-nn-trial-pred.tsv'
if sub.is_file():
    df = pd.read_csv(sub, sep='\t', header=None, names=['tID', 'Modifier', 'Head'])
    print('\n--- submission (en-nn-trial-pred.tsv, no header) ---')
    print(df.head())
    print('rows:', len(df))

### Nộp submission
1. File `en-nn-trial-pred.tsv` ở `/kaggle/working/submission/`.
2. Download rồi nộp ở challenge (3 cột `tID, Modifier, Head`, không header).

### Lưu ý
- **trial ρ là artifact n=2** — đừng dùng trial rho làm tín hiệu; xem **OOF ρ** (mean của `Mod`/`Head`) trong `metrics.json`.
- Muốn chạy lại chain với config khác: chỉnh `TRAIN5_SET`, ví dụ `TRAIN5_SET='--set,batch_size=16,--set,freeze_epochs=2'`. WARMUP thay đổi thì phải chạy lại `warmup` + `train5` + `predict` trong cùng session (ckpt ở `/kaggle/working` không tồn tại giữa các session).